In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

### Affinity translator

In [ ]:
import numpy as np

# Simulating real fMRI data
# 50 brain regions, monitored over 200 time points
# Some regions will intentionally "flash together"
np.random.seed(42)

# Generates random noise
brain_signals = np.random.randn(50, 200)

# Creates an artificial "Functional Network":
# We force Regions 0, 1, 2 and 3 to be identical (Strong Cluster)
brain_signals[1] = brain_signals[0] + np.random.normal(0, 0.1, 200)
brain_signals[2] = brain_signals[0] + np.random.normal(0, 0.1, 200)

In [ ]:
from interface.modeling_logic.translators import AffinityTranslator

# Initializes the Translator
# threshold=0.5: Ignores weak correlations
# regularization=0.5: Charges a fee of 0.5 energy to connect a node
translator = AffinityTranslator(threshold=0.5, regularization=0.5)

# Translate to Matrix Q
print("Translating brain signals to QUBO...")
q_matrix_rust = translator.from_time_series(brain_signals)

# Visualizing the first data to check
print("\n--- Excerpt from Matrix Q (First 4x4) ---")
for row in q_matrix_rust[:4]:
    print([f"{x:.2f}" for x in row[:4]])

print(f"\nMatrix ready to send to solver! Size: {len(q_matrix_rust)}x{len(q_matrix_rust)}")

In [ ]:
import torch

from interface.models.gnn import GNNBase

print("Running GNN to predict solution structure...")
model = GNNBase(len(q_matrix_rust))

# Simulated input (node ​​features)
input_features = torch.randn(1, len(q_matrix_rust))

with torch.no_grad():
    gnn_output = model(input_features)
    
priors_list = gnn_output.squeeze().tolist()

In [ ]:
import core_qyra_solver
sol = core_qyra_solver.solve(q_matrix_rust, priors_list, 1000)
print(sol)

### Conflict translator

In [6]:
from interface.modeling_logic.translators import ConflictTranslator
# Scenario: You have 5 possible meetings to attend today.
# But some of them happen at the same time (Conflict).

meetings_num = 5

# Weights: Some meetings are more important (Priority)
# Meeting 0 is worth a lot (3.0), the others are worth the normal amount (1.0)
importance = [3.0, 1.0, 1.0, 1.0, 1.0]

# List of Conflicts (Which meetings overlap in time)
# (0, 1): Meeting 0 and 1 are at the same time.
# (0, 2): Meeting 0 and 2 are at the same time.
# (3, 4): Meeting 3 and 4 are at the same time.
conflicts = [(0, 1), (0, 2), (3, 4)]

# Initialize the translator with a High Penalty
# Since meeting 0 is worth 3.0 and meeting 1 is worth 1.0, the sum is 4.0.
# The penalty must be GREATER than 4.0, otherwise it will choose both and "profit" from the balance.
translator = ConflictTranslator(penalty=5.0)

q_matrix = translator.from_conflict_list(meetings_num, conflicts, weights=importance)

print("--- Conflict Matrix (Independent Set) ---")
print("      R0     R1     R2     R3     R4")
for i, row in enumerate(q_matrix):
    row_str = "  ".join([f"{x:5.1f}" for x in row])
    print(f"R{i}: {row_str}")

print("\nInterpretation:")
print("- Negative values ​​on the diagonal: 'Please choose me!'")
print("- Values ​​of 5.0 outside the diagonal: 'NEVER choose us two together!'")

--- Conflict Matrix (Independent Set) ---
      R0     R1     R2     R3     R4
R0:  -3.0    5.0    5.0    0.0    0.0
R1:   5.0   -1.0    0.0    0.0    0.0
R2:   5.0    0.0   -1.0    0.0    0.0
R3:   0.0    0.0    0.0   -1.0    5.0
R4:   0.0    0.0    0.0    5.0   -1.0

Interpretation:
- Negative values ​​on the diagonal: 'Please choose me!'
- Values ​​of 5.0 outside the diagonal: 'NEVER choose us two together!'


In [7]:
import torch

from interface.models.gnn import GNNBase

print("Running GNN to predict solution structure...")
model = GNNBase(len(q_matrix))

# Simulated input (node ​​features)
input_features = torch.randn(1, len(q_matrix))

with torch.no_grad():
    gnn_output = model(input_features)
    
priors_list = gnn_output.squeeze().tolist()

Running GNN to predict solution structure...


In [8]:
import core_qyra_solver
sol = core_qyra_solver.solve(q_matrix, priors_list, 1000)
print(sol)

(b'\x01\x00\x00\x01\x00', -4.0)


Optimizing...
Optimization completed.
